# OpenWeather API - Data Exploration

## Part 1 - Setup & Connection

In [28]:
# Imports
import requests
import os
from dotenv import load_dotenv
import pandas as pd
import json
from datetime import datetime

# Load environment variables
load_dotenv()

# Get API key
api_key = os.getenv("OPENWEATHER_API_KEY")
print("API Key loaded:", api_key is not None)

API Key loaded: True


In [29]:
# Define parameters
city = "Paris"
url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"

# Call the API
response = requests.get(url)

# Check status
print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'coord': {'lon': 2.3488, 'lat': 48.8534}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 20.08, 'feels_like': 19.74, 'temp_min': 19.01, 'temp_max': 21.1, 'pressure': 1010, 'humidity': 61, 'sea_level': 1010, 'grnd_level': 1001}, 'visibility': 10000, 'wind': {'speed': 4.63, 'deg': 230}, 'clouds': {'all': 75}, 'dt': 1777825510, 'sys': {'type': 2, 'id': 2012208, 'country': 'FR', 'sunrise': 1777782424, 'sunset': 1777835267}, 'timezone': 7200, 'id': 2988507, 'name': 'Paris', 'cod': 200}


## Part 2 - Data Exploration

In [30]:
# Store the response
data = response.json()

In [31]:
# Explore key fields
print("=== LOCATION ===")
print("City:", data["name"])
print("Country:", data["sys"]["country"])
print("Coordinates:", data["coord"])

print("\n=== WEATHER ===")
print("Condition:", data["weather"][0]["main"])
print("Description:", data["weather"][0]["description"])

print("\n=== TEMPERATURE ===")
print("Temperature:", data["main"]["temp"], "°C")
print("Feels like:", data["main"]["feels_like"], "°C")
print("Min:", data["main"]["temp_min"], "°C")
print("Max:", data["main"]["temp_max"], "°C")

print("\n=== WIND ===")
print("Speed:", data["wind"]["speed"], "m/s")
print("Direction:", data["wind"]["deg"], "°")

print("\n=== VISIBILITY & CLOUDS ===")
print("Visibility:", data["visibility"], "m")
print("Cloud cover:", data["clouds"]["all"], "%")

=== LOCATION ===
City: Paris
Country: FR
Coordinates: {'lon': 2.3488, 'lat': 48.8534}

=== WEATHER ===
Condition: Clouds
Description: broken clouds

=== TEMPERATURE ===
Temperature: 20.08 °C
Feels like: 19.74 °C
Min: 19.01 °C
Max: 21.1 °C

=== WIND ===
Speed: 4.63 m/s
Direction: 230 °

=== VISIBILITY & CLOUDS ===
Visibility: 10000 m
Cloud cover: 75 %


## Part 3 - Geographic Coverage Analysis

In [32]:
import urllib.request
import gzip
import json
import os

# Download the city list from OpenWeather
url = "http://bulk.openweathermap.org/sample/city.list.json.gz"
output_path = "../data/raw/city_list.json.gz"

print("Downloading city list...")
urllib.request.urlretrieve(url, output_path)
print("Done !")

Done !


In [33]:
# Open and decompress the file
with gzip.open(output_path, "rb") as f:
    city_list = json.load(f)

print(f"Total cities: {len(city_list)}")
print("\nExample of one city:")
print(json.dumps(city_list[0], indent=2))

Total cities: 209579

Example of one city:
{
  "id": 833,
  "name": "\u1e28e\u015f\u0101r-e Sef\u012bd",
  "state": "",
  "country": "IR",
  "coord": {
    "lon": 47.159401,
    "lat": 34.330502
  }
}


In [34]:
# Convert to DataFrame for analysis
df_cities = pd.DataFrame(city_list)

print("Shape:", df_cities.shape)
print("\nCountries with most cities:")
print(df_cities["country"].value_counts().head(20))


Shape: (209579, 5)

Countries with most cities:
country
DE    28786
US    19972
FR    19965
ES    15439
IT     9878
RU     8768
CN     8715
ID     6590
AU     5593
PH     4962
GB     4882
CH     4145
PL     4100
AT     3682
IN     3634
BR     3621
CA     3302
PT     3231
RO     2872
MX     2426
Name: count, dtype: int64


In [35]:
# Major airport cities for the project
target_cities = {
    "Paris": "FR",
    "London": "GB", 
    "New York": "US",
    "Tokyo": "JP",
    "Dubai": "AE",
    "Frankfurt": "DE",
    "Amsterdam": "NL",
    "Madrid": "ES",
    "Rome": "IT",
    "Montreal": "CA",
    "Beijing": "CN",
    "Sydney": "AU",
    "Brussels": "BE"
}

# Filter cities from the list
selected = []
for city_name, country_code in target_cities.items():
    match = df_cities[
        (df_cities["name"] == city_name) & 
        (df_cities["country"] == country_code)
    ]
    if not match.empty:
        selected.append(match.iloc[0])
        print(f"✅ {city_name} ({country_code}) — id: {match.iloc[0]['id']}")
    else:
        print(f"❌ {city_name} ({country_code}) — NOT FOUND")

df_selected = pd.DataFrame(selected)
print(f"\nTotal selected: {len(df_selected)} cities")

✅ Paris (FR) — id: 2968815.0
✅ London (GB) — id: 2643743.0
✅ New York (US) — id: 5128638.0
✅ Tokyo (JP) — id: 1850147.0
✅ Dubai (AE) — id: 292223.0
❌ Frankfurt (DE) — NOT FOUND
✅ Amsterdam (NL) — id: 2759794.0
✅ Madrid (ES) — id: 3117735.0
✅ Rome (IT) — id: 3169070.0
❌ Montreal (CA) — NOT FOUND
✅ Beijing (CN) — id: 1816670.0
✅ Sydney (AU) — id: 2147714.0
✅ Brussels (BE) — id: 2800866.0

Total selected: 11 cities


In [36]:
# Search for Frankfurt in Germany
print("=== Frankfurt search ===")
print(df_cities[
    (df_cities["name"].str.contains("Frankfurt", case=False)) & 
    (df_cities["country"] == "DE")
]["name"].unique())

# Search for Montreal in Canada
print("\n=== Montreal search ===")
print(df_cities[
    (df_cities["name"].str.contains("Montreal", case=False)) & 
    (df_cities["country"] == "CA")
]["name"].unique())

=== Frankfurt search ===
<StringArray>
[    'Frankfurter Vorstadt',        'Frankfurt am Main',
         'Frankfurt (Oder)', 'Frankfurt Main Flughafen']
Length: 4, dtype: str

=== Montreal search ===
<StringArray>
['Montreal Lake', 'Montreal River']
Length: 2, dtype: str


In [37]:
# Search Montreal by name without country filter
print(df_cities[df_cities["name"].str.contains("Montr", case=False)]["name"].unique()[:10])

<StringArray>
[                              'Montrose',
                        'Real de Montroi',
                                'Montroy',
                             'Montricher',
                               'Montreux',
 'Arrondissement de Saint-Amand-Montrond',
                   'Saint-Amand-Montrond',
                                 'Montry',
                              'Montrouge',
                            'Montrottier']
Length: 10, dtype: str


In [38]:
target_cities = {
    "Paris": "FR",
    "London": "GB", 
    "New York": "US",
    "Tokyo": "JP",
    "Dubai": "AE",
    "Frankfurt am Main": "DE",
    "Amsterdam": "NL",
    "Madrid": "ES",
    "Rome": "IT",
    "Beijing": "CN",
    "Sydney": "AU",
    "Brussels": "BE"
}

In [39]:
# Search for Dorval in Canada
print(df_cities[
    (df_cities["name"].str.contains("Dorval", case=False)) & 
    (df_cities["country"] == "CA")
]["name"].unique())

<StringArray>
['Dorval']
Length: 1, dtype: str


In [40]:
# Real airport city names to verify
airport_check = {
    "Paris": ("CDG is in Roissy-en-France", "FR", "Roissy-en-France"),
    "London": ("Heathrow is in Hillingdon/London", "GB", "London"),
    "New York": ("JFK is in NYC", "US", "New York City"),
    "Tokyo": ("Narita is in Narita city", "JP", "Narita"),
    "Dubai": ("DXB is in Dubai", "AE", "Dubai"),
    "Frankfurt am Main": ("FRA is in Frankfurt", "DE", "Frankfurt am Main"),
    "Amsterdam": ("AMS is in Haarlemmermeer", "NL", "Haarlemmermeer"),
    "Madrid": ("MAD is in Barajas/Madrid", "ES", "Madrid"),
    "Rome": ("FCO is in Fiumicino", "IT", "Fiumicino"),
    "Beijing": ("PEK is in Shunyi", "CN", "Shunyi"),
    "Sydney": ("SYD is in Mascot/Sydney", "AU", "Sydney"),
    "Brussels": ("BRU is in Zaventem", "BE", "Zaventem"),
    "Montreal": ("YUL is in Dorval", "CA", "Dorval"),
}

# Search each real location
for main_city, (note, country, real_name) in airport_check.items():
    match = df_cities[
        (df_cities["name"] == real_name) & 
        (df_cities["country"] == country)
    ]
    status = f"✅ id: {match.iloc[0]['id']}" if not match.empty else "❌ NOT FOUND"
    print(f"{main_city:15} → {real_name:25} ({country}) {status}")

Paris           → Roissy-en-France          (FR) ✅ id: 2983073.0
London          → London                    (GB) ✅ id: 2643743.0
New York        → New York City             (US) ✅ id: 5128581.0
Tokyo           → Narita                    (JP) ✅ id: 2111684.0
Dubai           → Dubai                     (AE) ✅ id: 292223.0
Frankfurt am Main → Frankfurt am Main         (DE) ✅ id: 2925533.0
Amsterdam       → Haarlemmermeer            (NL) ❌ NOT FOUND
Madrid          → Madrid                    (ES) ✅ id: 3117735.0
Rome            → Fiumicino                 (IT) ✅ id: 6540287.0
Beijing         → Shunyi                    (CN) ✅ id: 2034754.0
Sydney          → Sydney                    (AU) ✅ id: 2147714.0
Brussels        → Zaventem                  (BE) ✅ id: 2783310.0
Montreal        → Dorval                    (CA) ✅ id: 5941925.0


In [41]:
print(df_cities[
    (df_cities["name"].str.contains("Haarlem", case=False)) & 
    (df_cities["country"] == "NL")
]["name"].unique())

<StringArray>
['Gemeente Haarlemmermeer', 'Gemeente Haarlem', 'Haarlem']
Length: 3, dtype: str


In [42]:
target_airports = {
    "Paris (CDG)":      {"city": "Roissy-en-France",       "country": "FR", "id": 2983073},
    "London (LHR)":     {"city": "London",                  "country": "GB", "id": 2643743},
    "New York (JFK)":   {"city": "New York City",           "country": "US", "id": 5128581},
    "Tokyo (NRT)":      {"city": "Narita",                  "country": "JP", "id": 2111684},
    "Dubai (DXB)":      {"city": "Dubai",                   "country": "AE", "id": 292223},
    "Frankfurt (FRA)":  {"city": "Frankfurt am Main",       "country": "DE", "id": 2925533},
    "Amsterdam (AMS)":  {"city": "Gemeente Haarlemmermeer", "country": "NL", "id": None},
    "Madrid (MAD)":     {"city": "Madrid",                  "country": "ES", "id": 3117735},
    "Rome (FCO)":       {"city": "Fiumicino",               "country": "IT", "id": 6540287},
    "Beijing (PEK)":    {"city": "Shunyi",                  "country": "CN", "id": 2034754},
    "Sydney (SYD)":     {"city": "Sydney",                  "country": "AU", "id": 2147714},
    "Brussels (BRU)":   {"city": "Zaventem",                "country": "BE", "id": 2783310},
    "Montreal (YUL)":   {"city": "Dorval",                  "country": "CA", "id": 5941925},
}

# Fetch id for Amsterdam
ams_match = df_cities[
    (df_cities["name"] == "Gemeente Haarlemmermeer") & 
    (df_cities["country"] == "NL")
]
target_airports["Amsterdam (AMS)"]["id"] = int(ams_match.iloc[0]["id"])

# Display summary statistics
print("=" * 65)
print(f"{'AIRPORT':<20} {'CITY':<28} {'CTY':<5} {'OW_ID'}")
print("=" * 65)
for airport, info in target_airports.items():
    print(f"{airport:<20} {info['city']:<28} {info['country']:<5} {info['id']}")

print("=" * 65)
print(f"Total airports: {len(target_airports)}")

# Stats by country
print("\n--- Countries ---")
countries = [v["country"] for v in target_airports.values()]
for c in sorted(set(countries)):
    count = countries.count(c)
    print(f"  {c} : {count} airport(s)")

AIRPORT              CITY                         CTY   OW_ID
Paris (CDG)          Roissy-en-France             FR    2983073
London (LHR)         London                       GB    2643743
New York (JFK)       New York City                US    5128581
Tokyo (NRT)          Narita                       JP    2111684
Dubai (DXB)          Dubai                        AE    292223
Frankfurt (FRA)      Frankfurt am Main            DE    2925533
Amsterdam (AMS)      Gemeente Haarlemmermeer      NL    2754999
Madrid (MAD)         Madrid                       ES    3117735
Rome (FCO)           Fiumicino                    IT    6540287
Beijing (PEK)        Shunyi                       CN    2034754
Sydney (SYD)         Sydney                       AU    2147714
Brussels (BRU)       Zaventem                     BE    2783310
Montreal (YUL)       Dorval                       CA    5941925
Total airports: 13

--- Countries ---
  AE : 1 airport(s)
  AU : 1 airport(s)
  BE : 1 airport(s)
  CA : 1 

## Part 4 - Data Quality Check (Airports Config)

In [43]:
print("=" * 50)
print("DATA QUALITY REPORT - Airports Config")
print("=" * 50)

# --- Check 1 : Missing IDs ---
print("\n[1] Missing IDs :")
missing = [(k, v) for k, v in target_airports.items() if v["id"] is None]
if missing:
    for airport, info in missing:
        print(f"   ❌ {airport}")
else:
    print("   ✅ No missing IDs")

# --- Check 2 : Duplicate IDs ---
print("\n[2] Duplicate IDs :")
ids = [v["id"] for v in target_airports.values()]
duplicates = [id_ for id_ in ids if ids.count(id_) > 1]
if duplicates:
    print(f"   ❌ Duplicates found: {set(duplicates)}")
else:
    print("   ✅ No duplicate IDs")

# --- Check 3 : Missing city or country ---
print("\n[3] Missing city or country :")
for airport, info in target_airports.items():
    if not info["city"] or not info["country"]:
        print(f"   ❌ {airport} : incomplete")
    else:
        print(f"   ✅ {airport} : OK")


DATA QUALITY REPORT - Airports Config

[1] Missing IDs :
   ✅ No missing IDs

[2] Duplicate IDs :
   ✅ No duplicate IDs

[3] Missing city or country :
   ✅ Paris (CDG) : OK
   ✅ London (LHR) : OK
   ✅ New York (JFK) : OK
   ✅ Tokyo (NRT) : OK
   ✅ Dubai (DXB) : OK
   ✅ Frankfurt (FRA) : OK
   ✅ Amsterdam (AMS) : OK
   ✅ Madrid (MAD) : OK
   ✅ Rome (FCO) : OK
   ✅ Beijing (PEK) : OK
   ✅ Sydney (SYD) : OK
   ✅ Brussels (BRU) : OK
   ✅ Montreal (YUL) : OK


In [45]:
# --- Check 4 : Live API test on ALL airports ---
print("\n[4] Live API test - All airports :")
print("-" * 70)

results = []  # Store results for quality check

for airport, info in target_airports.items():
    city_id = info["id"]
    url = (
        f"http://api.openweathermap.org/data/2.5/weather"
        f"?id={city_id}&appid={api_key}&units=metric"
    )
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        # Extract key fields
        temp        = data["main"].get("temp")
        humidity    = data["main"].get("humidity")
        wind_speed  = data["wind"].get("speed")
        wind_deg    = data["wind"].get("deg")
        visibility  = data.get("visibility")
        clouds      = data["clouds"].get("all")
        description = data["weather"][0].get("description")

        # Store in results list
        results.append({
            "airport"     : airport,
            "city"        : info["city"],
            "country"     : info["country"],
            "temp_c"      : temp,
            "humidity_pct": humidity,
            "wind_speed"  : wind_speed,
            "wind_deg"    : wind_deg,
            "visibility_m": visibility,
            "clouds_pct"  : clouds,
            "description" : description
        })

        print(f"✅ {airport:20} → {temp}°C | humidity: {humidity}% | wind: {wind_speed}m/s | {description}")

    else:
        print(f"❌ {airport:20} → Error {response.status_code}")

# Build DataFrame
df_weather = pd.DataFrame(results)

print("\n" + "=" * 70)
print("DATAFRAME PREVIEW")
print("=" * 70)
print(df_weather)

print("\n" + "=" * 70)
print("NULL VALUES CHECK")
print("=" * 70)
print(df_weather.isnull().sum())

print("\n" + "=" * 70)
print("BASIC STATISTICS")
print("=" * 70)
print(df_weather.describe())


[4] Live API test - All airports :
----------------------------------------------------------------------
✅ Paris (CDG)          → 18.96°C | humidity: 64% | wind: 5.66m/s | broken clouds
✅ London (LHR)         → 18.95°C | humidity: 57% | wind: 4.63m/s | overcast clouds
✅ New York (JFK)       → 11.97°C | humidity: 34% | wind: 10.29m/s | broken clouds
✅ Tokyo (NRT)          → 19.82°C | humidity: 85% | wind: 8.75m/s | overcast clouds
✅ Dubai (DXB)          → 33.96°C | humidity: 29% | wind: 4.63m/s | clear sky
✅ Frankfurt (FRA)      → 24.12°C | humidity: 35% | wind: 5.14m/s | clear sky
✅ Amsterdam (AMS)      → 16.68°C | humidity: 81% | wind: 3.6m/s | scattered clouds
✅ Madrid (MAD)         → 18.87°C | humidity: 52% | wind: 6.69m/s | few clouds
✅ Rome (FCO)           → 20.59°C | humidity: 52% | wind: 5.15m/s | broken clouds
✅ Beijing (PEK)        → 8.97°C | humidity: 46% | wind: 1m/s | clear sky
✅ Sydney (SYD)         → 17.44°C | humidity: 86% | wind: 2.57m/s | clear sky
✅ Brussels (BRU)  

# SAVE DATA - OpenWeather Exploration

In [46]:
# 1. Save weather DataFrame as CSV
timestamp = datetime.now().strftime("%Y%m%d")
csv_path = f"../data/raw/weather_airports_{timestamp}.csv"
df_weather.to_csv(csv_path, index=False)
print(f"✅ CSV saved : {csv_path}")

# 2. Save airports config as JSON
json_path = "../data/raw/airports_config.json"
with open(json_path, "w") as f:
    json.dump(target_airports, f, indent=4)
print(f"✅ JSON saved : {json_path}")

✅ CSV saved : ../data/raw/weather_airports_20260503.csv
✅ JSON saved : ../data/raw/airports_config.json
